# Phase 1 completion run: seed 44 (HI -> DE -> EN)

This notebook completes the **unfinished third seed** of the existing Phase 1 follow-up. It does not rerun or overwrite seeds 42/43, the original English -> Hindi -> German pilot, or the incomplete seed-44 files.

The prior seed-44 run saved only pretrained zero-shot baselines; it has no completed training stage. This notebook therefore starts seed 44 training from the pretrained model in a **new timestamped attempt folder**, using the same model, prompt, data-selection seeds, sample sizes, LoRA settings, and Hindi -> German -> English order. It updates `multiseed_summary.json` only after all three seeds have complete, validated matrices.

## Before running

1. In Colab, select a GPU runtime (T4 or better) and enable internet.
2. Accept access to `meta-llama/Llama-3.2-1B` and add a Colab secret named `HF_TOKEN`.
3. Keep the existing Google Drive results folder available: `MyDrive/continual_learning_phase1_hi_de_en/xnli-hi-de-en-multiseed-train1000-test500-20261110T045241Z`.
4. Upload the project file `phase1_xnli_multiseed_colab_package.zip` when prompted. It supplies the experiment/evaluator code.
5. Run cells in order. A failed attempt is retained; rerunning creates a new attempt folder. A completed attempt is detected and reused.

W&B logging is optional and disabled by default. This completes the three-seed **run**, not necessarily the scientific checkpoint: report the per-seed outcome without claiming reproduction if the criterion is not met.


In [ ]:
%pip -q install "datasets>=3.0" "transformers>=4.46" "peft>=0.13" accelerate bitsandbytes wandb


In [ ]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Select a GPU runtime in Colab, then rerun this cell.")

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token, add_to_git_credential=False)
del hf_token

ENABLE_WANDB = False
if ENABLE_WANDB:
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_MODE"] = "online"

print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
print("Hugging Face authentication completed; credentials were not printed.")


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import sys

PACKAGE_DIR = Path("/content/continual_learning_package")
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
print("Upload phase1_xnli_multiseed_colab_package.zip")
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one Phase 1 source package ZIP.")

archive_path = Path("/content") / zip_names[0]
with ZipFile(archive_path) as archive:
    root = PACKAGE_DIR.resolve()
    for member in archive.infolist():
        destination = (PACKAGE_DIR / member.filename).resolve()
        if destination != root and root not in destination.parents:
            raise ValueError("The uploaded archive contains an unsafe path.")
    archive.extractall(PACKAGE_DIR)

if not (PACKAGE_DIR / "shared" / "continual_learning_lora" / "experiment.py").is_file():
    raise FileNotFoundError("The uploaded ZIP is not the Phase 1 experiment package.")
sys.path.insert(0, str(PACKAGE_DIR))
print("Phase 1 experiment package loaded.")


In [ ]:
import json
import math
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

GROUP_ID = "xnli-hi-de-en-multiseed-train1000-test500-20261110T045241Z"
EXPERIMENT_ID = f"{GROUP_ID}-seed44"
LANGUAGES = ["hi", "de", "en"]
SEED = 44
TRAIN_SAMPLES_PER_LANGUAGE = 1000
TEST_SAMPLES_PER_LANGUAGE = 500
TEST_SPLIT_SEED = 42
DATASET_ID = "facebook/xnli"
MODEL_ID = "meta-llama/Llama-3.2-1B"

# Mount existing output Drive; do not generate a new experiment-group timestamp.
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/continual_learning_phase1_hi_de_en")
GROUP_ROOT = DRIVE_ROOT / GROUP_ID
INCOMPLETE_LEDGER = GROUP_ROOT / "seed-44" / "evaluation" / "evaluation_results.json"
SUMMARY_PATH = GROUP_ROOT / "multiseed_summary.json"
ATTEMPTS_ROOT = GROUP_ROOT / "seed-44-attempts"

if not INCOMPLETE_LEDGER.is_file():
    raise FileNotFoundError(f"Expected prior seed-44 ledger is missing: {INCOMPLETE_LEDGER}")
if not SUMMARY_PATH.is_file():
    raise FileNotFoundError(f"Expected prior multi-seed summary is missing: {SUMMARY_PATH}")

with INCOMPLETE_LEDGER.open(encoding="utf-8") as file:
    old_seed44 = json.load(file)
if old_seed44.get("experiment_id") != EXPERIMENT_ID:
    raise ValueError("The existing seed-44 ledger belongs to a different experiment.")
if old_seed44.get("task_order") != LANGUAGES:
    raise ValueError("The existing seed-44 ledger has a different task order.")
if old_seed44.get("run_metadata", {}).get("seed") != SEED:
    raise ValueError("The existing seed-44 ledger has a different seed.")
if any(value is not None for row in old_seed44.get("performance_matrix", []) for value in row):
    raise RuntimeError("Seed 44 already has stage results in its original ledger; inspect it before starting another run.")


def load_complete_run(path, seed):
    with Path(path).open(encoding="utf-8") as file:
        record = json.load(file)
    if record.get("experiment_id") != f"{GROUP_ID}-seed{seed}":
        raise ValueError(f"Wrong experiment ID in {path}.")
    if record.get("task_order") != LANGUAGES:
        raise ValueError(f"Wrong task order in {path}.")
    if record.get("run_metadata", {}).get("seed") != seed:
        raise ValueError(f"Wrong seed in {path}.")
    matrix = record.get("performance_matrix")
    if not isinstance(matrix, list) or len(matrix) != len(LANGUAGES):
        raise ValueError(f"Incomplete stage matrix in {path}.")
    if any(not isinstance(row, list) or len(row) != len(LANGUAGES) for row in matrix):
        raise ValueError(f"Malformed stage matrix in {path}.")
    if any(not isinstance(value, (int, float)) or not math.isfinite(value) for row in matrix for value in row):
        raise ValueError(f"The matrix contains missing or non-finite accuracy values in {path}.")
    metrics = record.get("metrics")
    if not isinstance(metrics, dict) or any(not isinstance(metrics.get(name), (int, float)) or not math.isfinite(metrics[name]) for name in ("ACC", "BWT", "FWT", "F")):
        raise ValueError(f"The metrics are incomplete in {path}.")
    baselines = record.get("baseline_accuracies")
    if not isinstance(baselines, list) or len(baselines) != len(LANGUAGES) or any(not isinstance(value, (int, float)) or not math.isfinite(value) for value in baselines):
        raise ValueError(f"The zero-shot baselines are incomplete in {path}.")
    return {
        "seed": seed,
        "experiment_id": record["experiment_id"],
        "matrix": matrix,
        "metrics": metrics,
        "evaluation_json": str(path),
    }


seed_results = [
    load_complete_run(GROUP_ROOT / f"seed-{seed}" / "evaluation" / "evaluation_results.json", seed)
    for seed in (42, 43)
]
print("Validated completed seed ledgers:", [item["seed"] for item in seed_results])
print("Prior seed-44 result is incomplete; the new run will use a separate attempt directory.")


In [ ]:
from datasets import load_dataset
from datetime import datetime, timezone

LABEL_NAMES = {0: "entailment", 1: "neutral", 2: "contradiction"}

def format_xnli_row(example):
    label_id = int(example["label"])
    if label_id not in LABEL_NAMES:
        raise ValueError(f"Unexpected XNLI label id: {label_id}")
    prompt = (
        "Task: Natural language inference\n"
        f"Premise: {example['premise']}\n"
        f"Hypothesis: {example['hypothesis']}\n"
        "Choose one label: entailment, neutral, contradiction\n"
        "Answer:"
    )
    return {"prompt": prompt, "target": " " + LABEL_NAMES[label_id]}

train_data = {}
test_data = {}
for language in LANGUAGES:
    train_raw = load_dataset(DATASET_ID, language, split="train")
    test_raw = load_dataset(DATASET_ID, language, split="test")
    train_count = min(TRAIN_SAMPLES_PER_LANGUAGE, len(train_raw))
    test_count = min(TEST_SAMPLES_PER_LANGUAGE, len(test_raw))
    if train_count != TRAIN_SAMPLES_PER_LANGUAGE or test_count != TEST_SAMPLES_PER_LANGUAGE:
        raise ValueError(f"Unexpected XNLI split size for {language}: train={train_count}, test={test_count}.")
    train_subset = train_raw.shuffle(seed=SEED).select(range(train_count))
    test_data[language] = test_raw.shuffle(seed=TEST_SPLIT_SEED).select(range(test_count))
    train_data[language] = train_subset.map(
        format_xnli_row,
        remove_columns=train_subset.column_names,
        desc=f"Formatting {language} training examples for seed {SEED}",
    )
    if set(train_data[language].column_names) != {"prompt", "target"}:
        raise ValueError(f"Unexpected formatted training columns for {language}.")
    print(f"{language}: train={len(train_data[language])}, fixed test={len(test_data[language])}")

print("Data prepared with the original seed-44 training selection and fixed test selection.")


In [ ]:
import gc
import statistics
from transformers import set_seed
from shared.continual_learning_lora import LlamaLoRATrainer, run_sequential_experiment

# Reuse a completed attempt after a runtime disconnect; ignore incomplete attempts.
ATTEMPTS_ROOT.mkdir(parents=True, exist_ok=True)
completed_attempts = []
for result_path in ATTEMPTS_ROOT.glob("*/evaluation/evaluation_results.json"):
    try:
        completed_attempts.append(load_complete_run(result_path, SEED))
    except (ValueError, KeyError, TypeError) as exc:
        print(f"Ignoring incomplete attempt {result_path}: {exc}")

if completed_attempts:
    seed44_result = sorted(completed_attempts, key=lambda item: item["evaluation_json"])[-1]
    print("Using already-completed seed-44 attempt:", seed44_result["evaluation_json"])
else:
    attempt_id = datetime.now(timezone.utc).strftime("attempt-%Y%m%dT%H%M%SZ")
    ATTEMPT_ROOT = ATTEMPTS_ROOT / attempt_id
    checkpoint_dir = ATTEMPT_ROOT / "checkpoints"
    evaluation_dir = ATTEMPT_ROOT / "evaluation"
    if ATTEMPT_ROOT.exists():
        raise FileExistsError(f"Attempt directory already exists; rerun to create a fresh attempt: {ATTEMPT_ROOT}")
    ATTEMPT_ROOT.mkdir(parents=True, exist_ok=False)

    set_seed(SEED)
    trainer = LlamaLoRATrainer(
        model_name=MODEL_ID,
        output_dir=checkpoint_dir,
        task_mode="causal_lm",
        max_length=512,
        learning_rate=2e-4,
        train_batch_size=1,
        eval_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        seed=SEED,
        lora_rank=8,
        lora_alpha=16,
        lora_dropout=0.05,
        load_in_4bit=True,
        gradient_checkpointing=True,
        token=True,
    )
    trainer.load_model_and_tokenizer()
    trainer.model.print_trainable_parameters()
    result = run_sequential_experiment(
        trainer,
        task_order=LANGUAGES,
        train_datasets=train_data,
        test_datasets=test_data,
        experiment_id=EXPERIMENT_ID,
        output_dir=evaluation_dir,
        dataset_name=DATASET_ID,
        task_name="xnli",
        run_metadata={
            "phase": 1,
            "method": "sequential_finetuning_no_replay",
            "seed": SEED,
            "seed_group": GROUP_ID,
            "test_split_seed": TEST_SPLIT_SEED,
            "model": MODEL_ID,
            "train_samples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
            "test_samples_per_language": TEST_SAMPLES_PER_LANGUAGE,
            "test_split": "test",
            "prompt_protocol": "phase1_b0_nli_prompt_candidate_label_log_likelihood",
            "attempt_id": attempt_id,
            "recovered_from_incomplete_seed44": True,
        },
        wandb_project=("asymmetric-interference-continual-learning" if ENABLE_WANDB else None),
        wandb_mode="online",
    )
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    seed44_result = load_complete_run(result["ledger"].json_path, SEED)
    del result
    print("Seed 44 completed:", seed44_result["evaluation_json"])

seed_results.append(seed44_result)
seed_results.sort(key=lambda item: item["seed"])
if [item["seed"] for item in seed_results] != [42, 43, 44]:
    raise ValueError("Expected exactly one validated result for seeds 42, 43, and 44.")


In [ ]:
import json
import os
import shutil
import statistics

matrices = [item["matrix"] for item in seed_results]
mean_matrix = [
    [statistics.mean(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
std_matrix = [
    [statistics.stdev(matrix[row][column] for matrix in matrices) for column in range(len(LANGUAGES))]
    for row in range(len(LANGUAGES))
]
seed_pattern_results = []
for item in seed_results:
    matrix = item["matrix"]
    final_row = matrix[-1]
    diagonal = [matrix[index][index] for index in range(len(LANGUAGES))]
    previous_final_mean = statistics.mean(final_row[:-1])
    previous_learned_mean = statistics.mean(diagonal[:-1])
    last_language_strong = final_row[-1] >= previous_final_mean
    earlier_languages_degraded = previous_final_mean < previous_learned_mean
    seed_pattern_results.append({
        "seed": item["seed"],
        "last_seen_language_strong_relative_to_previous_mean": last_language_strong,
        "earlier_languages_degraded_on_average": earlier_languages_degraded,
        "checkpoint_pattern_passed": last_language_strong and earlier_languages_degraded,
    })

final_mean = mean_matrix[-1]
first_seen_mean = [statistics.mean(matrix[index][index] for matrix in matrices) for index in range(len(LANGUAGES))]
report = {
    "experiment_group_id": GROUP_ID,
    "task_order": LANGUAGES,
    "seeds": [42, 43, 44],
    "train_examples_per_language": TRAIN_SAMPLES_PER_LANGUAGE,
    "test_examples_per_language": TEST_SAMPLES_PER_LANGUAGE,
    "fixed_test_split_seed": TEST_SPLIT_SEED,
    "mean_performance_matrix_rows_after_each_stage": mean_matrix,
    "std_performance_matrix_rows_after_each_stage": std_matrix,
    "mean_final_accuracies_by_language": dict(zip(LANGUAGES, final_mean)),
    "mean_first_seen_accuracies_by_language": dict(zip(LANGUAGES, first_seen_mean)),
    "per_seed_results": seed_results,
    "per_seed_checkpoint_checks": seed_pattern_results,
    "checkpoint_pattern_pass_rate": sum(item["checkpoint_pattern_passed"] for item in seed_pattern_results) / len(seed_pattern_results),
    "checkpoint_pattern_passed_all_seeds": all(item["checkpoint_pattern_passed"] for item in seed_pattern_results),
    "interpretation_note": "Three completed seeds provide a preliminary repeatability check, not a definitive scientific conclusion.",
}

# Preserve the previous summary before atomically replacing it.
summary_path = SUMMARY_PATH
backup_path = summary_path.with_name(
    f"multiseed_summary.pre-seed44-completion-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.json"
)
if backup_path.exists():
    raise FileExistsError(f"Refusing to overwrite summary backup: {backup_path}")
if summary_path.exists():
    shutil.copy2(summary_path, backup_path)
temporary_path = summary_path.with_suffix(".json.tmp")
temporary_path.write_text(json.dumps(report, indent=2, allow_nan=False) + "\n", encoding="utf-8")
os.replace(temporary_path, summary_path)

print("Updated three-seed summary:", summary_path)
print("Previous summary backup:", backup_path)
print("Per-seed checkpoint checks:", json.dumps(seed_pattern_results, indent=2))
print("Checkpoint pass rate:", report["checkpoint_pattern_pass_rate"])
if report["checkpoint_pattern_passed_all_seeds"]:
    print("All three seeds met the pilot criterion. This remains preliminary evidence, not definitive reproduction.")
else:
    print("Three-seed Phase 1 run is complete, but the checkpoint pattern was not met by every seed. Do not claim robust reproduction.")
